In [ ]:
import pandas as pd

df = pd.read_csv("C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes.csv")

# ler os primeiros registros do DataFrame
print(df.head().to_string())
# ler os últimos registros do DataFrame
print(df.tail().to_string())
# ver quantidades de linhas e colunas do DataFrame
print("Qtd: ", df.shape)
# ver os tipos de dados de cada coluna do DataFrame
print("tipos: ", df.dtypes)
# verificar se existem valores nulos no DataFrame
print("valores nulos: ", df.isnull().sum())

FUNCAO LAMBDA

In [26]:
# a funcao lambda é usada no lugar das funcoes (def), ela é mais simples e rapida porem tem limitacoes 
# é  apropriado usar em funcoes pequenas e com poucos argumentos.

df = pd.DataFrame({'numeros': [1, 2, 3, 4, 5]})
df['lambda'] = df['numeros'].apply(lambda x: x * 2)
print(df)

   numeros  lambda
0        1       2
1        2       4
2        3       6
3        4       8
4        5      10


LIMPEZA DE DADOS

In [ ]:
df = pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes.csv')
# configurar a exibicao do dataframe
pd.set_option('display.width', None)
print(df.head())

# remover dados
# df.drop(columns=['pais'], inplace=True) # outra forma de remover a coluna 'pais' do DataFrame
df.drop(labels=['pais'], axis=1, inplace=True) # remover a coluna 'pais' do DataFrame, axis=1 indica que é uma coluna, inplace=True para modificar o DataFrame original
df.drop(labels=7, axis=0, inplace=True) # remover a linha de indice 7 do DataFrame, axis=0 indica que é uma linha, inplace=True para modificar o DataFrame original
print(df.head())

df['nome'] = df['nome'].str.title() # converter os nomes para formato title (primeira letra maiuscula)
df['endereco'] = df['endereco'].str.upper # converter os endereços para formato upper (todas as letras maiusculas)
df['estado'] = df['estado'].str.lower() # converter os estados para formato lower (todas as letras minusculas)

# converter tipod de dados
df['idade'] = df['idade'].astype(int) # converter a coluna 'idade' para tipo inteiro

# tratar valores nulos
df_fillna = df.fillna(0) # preencher os valores nulos com 0
df_dropna = df.dropna() # remover as linhas com valores nulos do DataFrame
df_dropna4 = df.dropna(thresh=4) # remover as linhas com menos de 4 valores não nulos do DataFrame
df = df.dropna(subset=['cpf']) # remover as linhas com valores nulos na coluna 'cpf' do DataFrame

print("Valores nulos: ", df.isnull().sum())

# outro forma de tratar valores nulos
df.fillna({'cpf': 'invalido'}, inplace=True) # preencher os valores nulos da coluna 'cpf' com 'invalido'
df['endereco'] = df['endereco'].fillna('endereco nao inserido') # preencher os valores nulos da coluna 'endereco' com 'endereco nao inserido'
df['idade_corrigida'] = df['idade'].fillna(df['idade'].mean()) # preencher os valores nulos da coluna 'idade' com a media da coluna 'idade'

# tratar formato de dados
df['data_corrigida'] = pd.to_datetime(df['data'], format='%d/%m/%Y', errors='coerce') # converter a coluna 'data' para formato datetime, format especifica o formato da data, errors='coerce' para converter os valores inválidos para NaT (Not a Time)

# tratar dados duplicados   
print("Qtd de linhas antes de remover duplicados: ", df.shape[0])
df.drop_duplicates()
df.drop_duplicates(subset='cpf', inplace=True) # remover as linhas duplicadas com base na coluna 'cpf', inplace=True para modificar o DataFrame original
print("Qtd de linhas depois de remover duplicados: ", len(df))
print('dados limpos:\n ', df)






salvar dados

In [ ]:
df['data'] = df['data_corrigida'] # substituir a coluna 'data' pela coluna 'data_corrigida'
df['idade'] = df['idade_corrigida'] # substituir a coluna 'idade' pela coluna 'idade_corrigida'

df_salvar = df[['nome', 'cpf', 'data', 'idade', 'endereco', 'estado']] # selecionar as colunas para salvar
df_salvar.to_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_limpeza.csv', index=False) # salvar o DataFrame tratado em um novo arquivo CSV, index=False para não salvar o índice do DataFrame no arquivo CSV

print('Novo dataframe: \n', pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_limpeza.csv'))
      
      

OUTLIES SÃO VALORES INCONSISTENTES NA BASE (MUITO ALTOS OU BAIXOS)

In [ ]:
from scipy import stats

pd.set_option('display.width', None)
df = pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_limpeza.csv')
# filtror os clientes com idade maior que 100 anos
df_filtro = df[df['idade'] > 100]

print(df_filtro[['nome', 'idade']])

# identificar outliers com zscore
scores = stats.zscore(df['idade'])
outlies = df[scores >= 3]
print(outlies)

# filtrar outliers com zscore
df_score = df[(stats.zscore(df['idade']) < 3)]


In [ ]:
# identificar outliers com IQR
q1 = df['idade'].quantile(0.25) # calcular o primeiro quartil (Q1) da coluna idade
q3 = df['idade'].quantile(0.75) # calcular o terceiro quartil (Q3) da coluna idade
iqr = q3 - q1 # calcular o intervalo interquartil (IQR)

limite_baixo = q1 - 1.5 * iqr # calcular o limite inferior para identificar outliers
limite_alto = q3 + 1.5 * iqr # calcular o limite superior para identificar outliers
print("limite iqr: ", limite_baixo,",", limite_alto)

outlies_iqr = df[(df['idade'] < limite_baixo) | (df['idade'] > limite_alto)] # identificar os outliers com base nos limites calculados
print(outlies_iqr)


# filtrar outliers com IQR
df_iqr = df[(df['idade'] >= limite_baixo) & (df['idade'] <= limite_alto)]
print(df_iqr)

In [ ]:
# filtrar enderecos invalidos
df['endereco'] = df['endereco'].apply(lambda x: 'Endereço invalido' if len(x.split('\n')) < 3 else x) # aplicar a função lambda para verificar se o endereço tem menos de 3 linhas, se sim, substituir por 'Endereço invalido'
print((df['endereco'] == 'Endereço invalido').sum()) # contar quantos endereços foram classificados como 'Endereço invalido'


# tratar campos de texto
df['nome'] = df['nome'].apply(lambda x: 'Nome invalido' if isinstance(x, str) and len(x) > 50  else x) # aplicar a função lambda para verificar se o nome é uma string e tem mais de 50 caracteres, se sim, substituir por 'Nome invalido', caso contrário, manter o valor original
print((df['nome'] == 'Nome invalido').sum())

print('dados com tratamento de outliers e campos de texto:\n', df)

# salvar o DataFrame tratado em um novo arquivo CSV
df.to_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_remove_outlies.csv', index=False) # salvar o DataFrame tratado em um novo arquivo CSV, index=False para não salvar o índice do DataFrame no arquivo CSV

In [78]:
import numpy as np

pd.set_option('display.width', None) # configurar a largura de exibição do DataFrame para None, permitindo que ele se ajuste automaticamente à largura do conteúdo
pd.set_option('display.max_columns', None) # configurar o número máximo de colunas a serem exibidas para None, permitindo que todas as colunas sejam exibidas

df = pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_remove_outlies.csv')
print(df.head())

# mascarar dados
df['cpf_mascara'] = df['cpf'].apply(lambda cpf: f'{cpf[:3]}.***.***.{cpf[-2:]}') # aplicar a função lambda para mascarar o CPF, mantendo os 3 primeiros e os 2 últimos dígitos, e substituindo os dígitos do meio por asteriscos

# corrigir datas
data_atual = pd.to_datetime('today') # obter a data atual
df['data_corrigida'] = df['data'].where(df['data'] <= data_atual, pd.to_datetime('1900-01-01')) # corrigir as datas, mantendo as datas válidas e substituindo as datas futuras por 'data invalida'
df['idade_corrigida'] = df[data_atual].year - df['data_corrigida'].dt.year # corrigir as idades, calculando a idade com base na data corrigida e na data atual
df['idade_corrigida'] -= ((data_atual.month <= df['data_corrigida'].dt.month) & (data_atual.day < df['data_corrigida'].dt.day)).astype(int) # ajustar a idade corrigida, subtraindo 1 se a data atual for antes do aniversário da data corrigida
df.loc[df['idade_corrigida'] > 100, 'idade_corrigida'] = np.NaN # substituir as idades corrigidas maiores que 100 por NaN (Not a Number)


                      nome             cpf        data  idade  \
0  Sr. Vitor Gabriel Silva  640.719.385-01  1933-12-06     90   
1            Calebe Santos  651.297.384-46  1984-03-06     40   
2             Evelyn Pires  107.643.528-90  1944-03-09     80   
3       Maria Julia Castro  164.750.293-43  1947-04-12     76   
4           Arthur Da Rosa  085.167.249-30  1970-04-16     53   

            endereco         estado     cpf_mascara       data_corrigida  \
0  Endereço invalido  nao informado  640.***.***.01  1900-01-01 00:00:00   
1  Endereço invalido  nao informado  651.***.***.46  1900-01-01 00:00:00   
2  Endereço invalido  nao informado  107.***.***.90  1900-01-01 00:00:00   
3  Endereço invalido  nao informado  164.***.***.43  1900-01-01 00:00:00   
4  Endereço invalido  nao informado  085.***.***.30  1900-01-01 00:00:00   

                 rua         bairro  
0  Endereço invalido  nao informado  
1  Endereço invalido  nao informado  
2  Endereço invalido  nao informado  


TypeError: '>=' not supported between instances of 'Timestamp' and 'str'

In [ ]:
# separar o endereco
df['rua'] = df['endereco'].apply(lambda x: x.split('\n')[0].strip()) # aplicar a função lambda para separar o endereço, pegando a primeira linha do endereço e removendo os espaços em branco no início e no final
df['bairro'] = df['endereco'].apply(lambda x: x.split('\n')[1].strip() if len(x.split('\n')) > 1 else 'nao informado') # aplicar a função lambda para separar o endereço, pegando a segunda linha do endereço e removendo os espaços em branco no início e no final, se existir, caso contrário, atribuir 'nao informado' ao bairro
df['estado'] = df['endereco'].apply(lambda x: x.split(' / ')[-1].strip() if len(x.split('\n')) > 1 else 'nao informado') # aplicar a função lambda para separar o endereço, pegando a última parte do endereço após a barra e removendo os espaços em branco no início e no final, se existir, caso contrário, atribuir 'nao informado' ao estado

# verificando o formato do endereco
df['rua'] = df['rua'].apply(lambda x: 'endereco invalido' if len(x) > 50 or len(x) < 5 else x) # aplicar a função lambda para verificar se a rua tem mais de 50 caracteres ou menos de 5 caracteres, se sim, substituir por 'endereco invalido', caso contrário, manter o valor original 


# corrigir dados errados
df['cpf'] = df['cpf'].apply(lambda x: x if len(x) == 14 else 'cpf invalido') # aplicar a função lambda para verificar se o CPF tem 14 caracteres, se sim, manter o valor original, caso contrário, substituir por 'cpf invalido'

estados_br = ['ac', 'al', 'ap', 'am', 'ba', 'ce', 'df', 'es', 'go', 'ma', 'mt', 'ms', 'mg', 'pa', 'pb', 'pr', 'pe', 'pi', 'rj', 'rn', 'rs', 'ro', 'ra', 'sc', 'sp', 'se', 'to'] # lista de estados do Brasil em formato lowercase
df['estado'] = df['estado'].str.upper().apply(lambda x: x if x in estados_br else 'desconhecido')
print(df.head())

# modificar as colunas originais com os dados corrigidos
df['cpf'] = df['cpf_mascara']
df['idade'] = df['idade_corrigida']
df['endereco'] = df['rua']
df['estado'] = df['estado']
df_salvar.to_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_tratados.csv', index=False) # salvar o DataFrame tratado em um novo arquivo CSV, index=False para não salvar o índice do DataFrame no arquivo CSV




ANALISE ESPLORATORIA DE DADOS (AED)

In [32]:
df = pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes-v2.csv') 

print(df.head().to_string()) 
print(df.tail().to_string())

df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y', errors='coerce')

print('Verificação inicial: ')
print(df.info())

                  nome             cpf  idade        data                                                          endereco               estado    pais   salario      nivel_educacao  numero_filhos estado_civil  anos_experiencia area_atuacao
0           Vitor Lima  612.570.493-99     56  19/01/1968           Estrada Barbosa, 255\nVila Tirol\n53133-647 Barros / PR                 Pará  Brasil  13550.54        Ensino Médio              0       Casado                17   Tecnologia
1    Ana Clara Martins  648.129.530-06     33  02/12/1990        Favela Duarte, 44\nBonfim\n37709685 Farias das Flores / SP       Santa Catarina  Brasil  10645.16                 NaN              3          NaN                15          NaN
2      Heloísa da Mata  340.981.256-33     49  31/01/1975          Pátio de Vieira, 83\nJonas Veiga\n53832-319 Cardoso / PR  Rio Grande do Norte  Brasil   3124.72        Ensino Médio              3       Casado                 3    Indústria
3          Juan Novaes  097.168.

In [41]:
print('Analise de dados nulos:\n', df.isnull().sum())
print("% de dados nulos:\n", df.isnull().mean() * 100)
df.dropna(inplace=True)


Analise de dados nulos:
 nome                0
cpf                 0
idade               0
data                0
endereco            0
estado              0
pais                0
salario             0
nivel_educacao      0
numero_filhos       0
estado_civil        0
anos_experiencia    0
area_atuacao        0
dtype: int64
% de dados nulos:
 nome                0.0
cpf                 0.0
idade               0.0
data                0.0
endereco            0.0
estado              0.0
pais                0.0
salario             0.0
nivel_educacao      0.0
numero_filhos       0.0
estado_civil        0.0
anos_experiencia    0.0
area_atuacao        0.0
dtype: float64


In [ ]:
print('remoção de dados nulos:\n', df.isnull().sum().sum())

print('Dados duplicados: ', df.duplicated().sum())

print('Dados unicos: ', df.nunique())

print('Estatisticas dos dados: ', df.describe()) # exibir as estatísticas descritivas dos dados numéricos do DataFrame, como contagem, média, desvio padrão, valores mínimos e máximos, e os quartis (25%, 50% e 75%) para cada coluna numérica do DataFrame.



remoção de dados nulos:
 0
Dados duplicados:  0
Dados unicos:  nome                7032
cpf                 8620
idade                 74
data                6875
endereco            8629
estado                27
pais                   1
salario             8581
nivel_educacao         4
numero_filhos          6
estado_civil           4
anos_experiencia      68
area_atuacao           5
dtype: int64
Estatisticas dos dados:               idade                        data       salario  numero_filhos  \
count  8629.000000                        8629   8629.000000    8629.000000   
mean     40.012516  1983-10-06 07:38:25.017962   6771.998985       1.000811   
min      18.000000         1932-04-24 00:00:00   1357.960000       0.000000   
25%      27.000000         1973-05-20 00:00:00   4232.500000       0.000000   
50%      38.000000         1985-07-26 00:00:00   6061.630000       1.000000   
75%      50.000000         1996-12-14 00:00:00   8449.750000       2.000000   
max      91.000000   

In [48]:
df = df[['idade', 'data', 'estado', 'salario', 'nivel_educacao', 'numero_filhos', 'estado_civil', 'area_atuacao']]
print(df.head().to_string())

df.to_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_tratados_v2.csv', index=False)

   idade       data               estado   salario      nivel_educacao  numero_filhos estado_civil area_atuacao
0     56 1968-01-19                 Pará  13550.54        Ensino Médio              0       Casado   Tecnologia
2     49 1975-01-31  Rio Grande do Norte   3124.72        Ensino Médio              3       Casado    Indústria
3     54 1969-08-26         Minas Gerais   7534.60        Ensino Médio              0       Casado   Tecnologia
4     61 1963-03-19                Amapá   4067.73  Ensino Fundamental              1     Solteiro     Comércio
5     30 1993-12-09              Alagoas   6809.94        Ensino Médio              1       Casado   Tecnologia


In [ ]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler # importar as classes RobustScaler, MinMaxScaler e StandardScaler do módulo sklearn.preprocessing para realizar a normalização e padronização dos dados numéricos

In [58]:
pd.set_option('Display.width', None)
pd.set_option('display.max_columns', None)

df = pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_tratados_v2.csv')

df = df[['idade', 'salario']]
print(df.head())

   idade   salario
0     56  13550.54
1     49   3124.72
2     54   7534.60
3     61   4067.73
4     30   6809.94


NORMALIZAÇÃO - MINMAXSCALER

In [ ]:
scaler = MinMaxScaler() # criar um objeto MinMaxScaler para escalar os dados numéricos para o intervalo de 0 a 1
df['idade_scaler'] = scaler.fit_transform(df[['idade']]) #  aplicar o MinMaxScaler para escalar a coluna 'idade' para o intervalo de 0 a 1, e armazenar os valores escalados em uma nova coluna 'idade_scaler'
df['salario_scaler'] = scaler.fit_transform(df[['salario']]) # aplicar o MinMaxScaler para escalar a coluna 'salario' para o intervalo de 0 a 1, e armazenar os valores escalados em uma nova coluna 'salario_scaler'

min_max_scaler = MinMaxScaler(feature_range=(-1, 1))  # criar um objeto MinMaxScaler com o intervalo de escala definido para -1 a 1
df['idade_min_max'] = min_max_scaler.fit_transform(df[['idade']]) # aplicar o MinMaxScaler para escalar a coluna 'idade' para o intervalo de -1 a 1, e armazenar os valores escalados em uma nova coluna 'idade_min_max'
df['salario_min_max'] = min_max_scaler.fit_transform(df[['salario']]) # aplicar o MinMaxScaler para escalar a coluna 'salario' para o intervalo de -1 a 1, e armazenar os valores escalados em uma nova coluna 'salario_min_max'


PADRONIZAÇÃO - STANDARDSCALER ROBUSTSCALER

In [64]:
scaler = StandardScaler() # criar um objeto StandardScaler para padronizar os dados numéricos, ajustando os valores para terem média 0 e desvio padrão 1
df['idade_standard'] = scaler.fit_transform(df[['idade']]) # aplicar o StandardScaler para padronizar a coluna 'idade', ajustando os valores para terem média 0 e desvio padrão 1, e armazenar os valores padronizados em uma nova coluna 'idade_standard'
df['salario_standard'] = scaler.fit_transform(df[['salario']]) # aplicar o StandardScaler para padronizar a coluna 'salario', ajustando os valores para terem média 0 e desvio padrão 1, e armazenar os valores padronizados em uma nova coluna 'salario_standard'

scaler = RobustScaler()
df['idade_robust'] = scaler.fit_transform(df[['idade']])
df['salario_robust'] = scaler.fit_transform(df[['salario']])

print(df.head().to_string())


   idade   salario  idade_scaler  salario_scaler  idade_min_max  salario_min_max  idade_standard  salario_standard  idade_robust  salario_robust
0     56  13550.54      0.520548        0.429638       0.041096        -0.140724        1.038112          1.949011      0.782609        1.775780
1     49   3124.72      0.424658        0.062257      -0.150685        -0.875487        0.583582         -1.048690      0.478261       -0.696404
2     54   7534.60      0.493151        0.217650      -0.013699        -0.564699        0.908246          0.219268      0.695652        0.349273
3     61   4067.73      0.589041        0.095486       0.178082        -0.809028        1.362775         -0.777549      1.000000       -0.472796
4     30   6809.94      0.164384        0.192115      -0.671233        -0.615770       -0.650140          0.010909     -0.347826        0.177440


CODIFICAÇÃO DE VARIAVEIS - TRANSFORMAR TEXTO EM NUMERO

In [ ]:
from sklearn.preprocessing import LabelEncoder # importar a classe LabelEncoder do módulo sklearn.preprocessing para codificar variáveis categóricas em valores numéricos


In [ ]:
pd.set_option('display.width', None)

df = pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_tratados_v2.csv')

#codificação one-not para estado civil
df = pd.concat([df, pd.get_dummies(df['estado_civil'], prefix='estado_civil')], axis=1) # usar a função pd.get_dummies para criar variáveis dummy (variáveis binárias) a partir da coluna 'estado_civil', prefix='estado_civil' para adicionar um prefixo às novas colunas criadas, axis=1 para concatenar as novas colunas ao DataFrame original na horizontal

# codificação ordinal para nivel de educacao
educacao_ordem = {'Ensino Fundamental': 1, 'Ensino Médio': 2, 'Ensino Superior': 3, 'Pós-Graduação': 4} # criar um dicionário para mapear os níveis de educação para valores numéricos ordinais
df['nivel_educacao_ordinal'] = df['nivel_educacao'].map(educacao_ordem) # usar a função map para aplicar o mapeamento do dicionário 'educacao_ordem' à coluna 'nivel_educacao', criando uma nova coluna 'nivel_educacao_ordinal' com os valores numéricos correspondentes


# transformar area atuacao usando o cat codes
df['area_atuacao_cat'] = df['area_atuacao'].astype('category').cat.codes # converter a coluna 'area_atuacao' para o tipo 'category' e usar a função cat.codes para atribuir códigos numéricos a cada categoria, criando uma nova coluna 'area_atuacao_cat' com os códigos numéricos correspondentes


# label encoding para estado
label_encoder = LabelEncoder() # criar um objeto LabelEncoder para codificar a coluna 'estado' em valores numéricos
df['estado_label'] = label_encoder.fit_transform((df['estado'])) # usar a função fit_transform do LabelEncoder para codificar a coluna 'estado' em valores numéricos, criando uma nova coluna 'estado_label' com os valores numéricos correspondentes
print(df.head().to_string())    


   idade        data               estado   salario      nivel_educacao  numero_filhos estado_civil area_atuacao  estado_civil_Casado  estado_civil_Divorciado  estado_civil_Solteiro  estado_civil_Viúvo  nivel_educacao_ordinal  area_atuacao_cat  estado_label
0     56  1968-01-19                 Pará  13550.54        Ensino Médio              0       Casado   Tecnologia                 True                    False                  False               False                     2.0                 4            15
1     49  1975-01-31  Rio Grande do Norte   3124.72        Ensino Médio              3       Casado    Indústria                 True                    False                  False               False                     2.0                 2            18
2     54  1969-08-26         Minas Gerais   7534.60        Ensino Médio              0       Casado   Tecnologia                 True                    False                  False               False                     2.0 

In [80]:
from scipy import stats
import numpy as np

TRANSFORMACAO FEATURES

In [87]:
# transformacao logaritmica
df = pd.read_csv('C:\\Users\\jv001\\Downloads\\GitHub\\curso_ebac\\data\\raw\\clientes_tratados_v2.csv')
df['salario_log'] = np.log1p(df['salario']) # log1p é usado para aplicar a transformação logarítmica, adicionando 1 ao valor original para evitar problemas com valores zero ou negativos, e armazenar os valores transformados em uma nova coluna 'salario_log'

# transformacao box cox
df['salario_boxcox'], _ = stats.boxcox(df['salario'] + 1 ) # aplicar a transformação Box-Cox à coluna 'salario', adicionando 1 para evitar problemas com valores zero ou negativos, e armazenar os valores transformados em uma nova coluna 'salario_boxcox'

# verificando a frequencia para estado
estado_freq = df['estado'].value_counts() / len(df) # calcular a frequência relativa de cada categoria na coluna 'estado' dividindo o número de ocorrências de cada categoria pelo número total de registros no DataFrame
df['estado_freq'] = df['estado'].map(estado_freq) # usar a função map para aplicar o mapeamento da frequência relativa à coluna 'estado', criando uma nova coluna 'estado_freq' com os valores de frequência correspondentes

#interacoes
df['interacao_idade_filho'] = df['idade'] * df['numero_filhos'] # criar uma nova coluna 'interacao_idade_filho' que é o resultado da multiplicação da coluna 'idade' pela coluna 'numero_filhos', representando a interação entre essas duas variáveis
print(df.head().to_string())

   idade        data               estado   salario      nivel_educacao  numero_filhos estado_civil area_atuacao  salario_log  salario_boxcox  estado_freq  interacao_idade_filho
0     56  1968-01-19                 Pará  13550.54        Ensino Médio              0       Casado   Tecnologia     9.514255       10.182574     0.038127                      0
1     49  1975-01-31  Rio Grande do Norte   3124.72        Ensino Médio              3       Casado    Indústria     8.047420        8.522195     0.037316                    147
2     54  1969-08-26         Minas Gerais   7534.60        Ensino Médio              0       Casado   Tecnologia     8.927394        9.514150     0.036737                      0
3     61  1963-03-19                Amapá   4067.73  Ensino Fundamental              1     Solteiro     Comércio     8.311086        8.818123     0.036968                     61
4     30  1993-12-09              Alagoas   6809.94        Ensino Médio              1       Casado   Tecnolog